### Establish connection for scraping

In [ ]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

url = "https://basscentral.com/"
response = requests.get(url, headers=headers, timeout=10)

response.status_code

In [ ]:
html_doc = response.text

from bs4 import BeautifulSoup
soup = BeautifulSoup(html_doc, 'html.parser')
soup.prettify()

In [ ]:
main_categories = soup.find("div", class_="navProducts-rootMenu-list")

In [ ]:
catg = main_categories.find_all("li", class_="navProducts-item")
categories = {}
for cat in catg:
    cat_name = cat.find("a").get_text(strip=True)
    cat_link = cat.find("a").get("href")
    categories[cat_name] = cat_link

categories

In [ ]:
category_response = requests.get(categories['Basses'], headers=headers, timeout=10)
category_soup = BeautifulSoup(category_response.text, 'html.parser')
category_soup.prettify()

In [ ]:
products_container = category_soup.find("div", class_="product-listing-form")
products_container

In [ ]:
product_card = products_container.find_all("li", class_="product")
product_card

In [ ]:
product_info = product_card[1].find("div", class_="card-body")
product_info

In [ ]:
product_name = product_info.find("h3").get_text(strip=True)
product_price = product_info.find("div", class_="card-text card-text--price").get_text(strip=True)


In [ ]:
product_name

### Full data scraping with image

In [ ]:
import os
from urllib.parse import urlparse

target_pages = 2

products = {

}

for cat, link in categories.items():
    for page in range(1, target_pages+1):
        dynamic_url = f"{link}?page={page}" # Dynamic url to iterate through categories and its pages
        
        html_doc = requests.get(dynamic_url, headers=headers, timeout=10) #Establish connection
        if html_doc.status_code != 200: #Break loop if the request throws an error and move on to the next page
            break

        category_soup = BeautifulSoup(html_doc.text, 'html.parser') # Change html_doc plain text to BeautifulSoup Object
        
        try:
            products_container = category_soup.find("div", class_="product-listing-form") # Select product container of each page
            print(f"Scraping page {page} of {cat}")
        except Exception as e:
            print(f"No product container found in page {page} for {cat}")
            continue # Skip to the next page if product container not found

        try:
            product_card = products_container.find_all("li", class_="product") # Select all occurences of product card inside the product container
        except Exception as e:
            print(f"No products for {cat}")
            continue

        for pro in product_card: # For loop to append each entry inside the products dictionary
            product_name = pro.find("h3").get_text(strip=True)
            product_price = pro.find("div", class_="card-text card-text--price").get_text(strip=True)

            img_url = pro.find("img").get("src")

            try:
                img_data = requests.get(img_url, headers=headers, stream=True)
                img_data.raise_for_status()

                parsed_url = urlparse(img_url)
                ext = os.path.splitext(parsed_url.path)[1]

                if not ext:
                    ext = ".jpg"

                file_name = product_name.replace("/", ",")
                folder_path = f"/home/chris/Documents/Broadway Python/Python Project I/Images"
                filename = f"{file_name}{ext}"
                full_file_path = os.path.join(folder_path, filename)

                with open(full_file_path, "wb") as file:
                    for chunk in img_data.iter_content(chunk_size=8192):
                        file.write(chunk)
                
                print(f"Successfully downloaded to {full_file_path}")

            except Exception as e:
                print(f"Skipping {img_url} due to error: {e}")

            if cat not in products:
                products[cat] = []

            products[cat].append({
                'name': product_name,
                'price': product_price,
                'image_url': full_file_path
            })

print("Scraping Complete!!")

Scraping page 1 of Basses
Scraping page 2 of Basses
Scraping page 1 of Strings
Scraping page 2 of Strings
Scraping page 1 of Amps
Scraping page 2 of Amps
Scraping page 1 of Accessories
Scraping page 2 of Accessories
Scraping page 1 of Effects & Pedals
Scraping page 2 of Effects & Pedals
Scraping page 1 of Bargain Basement
No products for Bargain Basement
Scraping page 2 of Bargain Basement
No products for Bargain Basement
Scraping Complete!!


In [214]:
products

{'Basses': [{'name': 'MTD U.S.A. 635-24, Clear to Blue Burst / Rosewood *Similar On Order, ETA  Jan 2027',
   'price': '$8,899.00',
   'image_url': '/home/chris/Documents/Broadway Python/Python Project I/Images/MTD U.S.A. 635-24, Clear to Blue Burst , Rosewood *Similar On Order, ETA  Jan 2027.jpg'},
  {'name': 'Sandberg California Martin Mendez Signature V4TT, Soft Aged White / Ebony *On Order ETA Sept 2026*',
   'price': '$3,299.00',
   'image_url': '/home/chris/Documents/Broadway Python/Python Project I/Images/Sandberg California Martin Mendez Signature V4TT, Soft Aged White , Ebony *On Order ETA Sept 2026*.JPG'},
  {'name': 'Dingwall NG2-to-NG3 Upgrade Kit (5-String)',
   'price': '$337.00',
   'image_url': '/home/chris/Documents/Broadway Python/Python Project I/Images/Dingwall NG2-to-NG3 Upgrade Kit (5-String).jpg'},
  {'name': "MTD Kingston Super 4 (V2), Dr. Brown's Burst with Maple",
   'price': '$1,199.00',
   'image_url': "/home/chris/Documents/Broadway Python/Python Project I/

In [215]:
import json
with open("products.json", "w") as f:
    json.dump(products, f, indent=4)